In [ ]:
from google.colab import drive
import sys
drive.mount('/content/drive')
%cd /content
!git clone https://github.com/arokah25/mixture_of_experts_project.git
%cd mixture_of_experts_project
!{sys.executable} -m pip install -U pip setuptools wheel
!{sys.executable} -m pip install -r requirements.txt
!{sys.executable} -m pip install -e .

!git branch
import moe
print("Imported moe from:", moe.__file__)



In [ ]:
CKPT_ROOT = "/content/drive/MyDrive/moe_project/checkpoints"

import os
os.makedirs(CKPT_ROOT, exist_ok=True)
CKPT_ROOT


### Train dense model
result: /content/drive/MyDrive/moe_project/checkpoints/Dense/E50/
    summary.json
    model.pt
    metrics.pt

In [ ]:
#Note 512 is default width for dense model
#E50
!python scripts/train_cifar10.py \
    --FF_layer Dense \
    --epochs 50 \
    --ff_width 512 \
    --batch_size 128 \
    --ckpt_root "{CKPT_ROOT}"


### Train softmoe
result: /content/drive/MyDrive/moe_project/checkpoints/SoftMoE/E50-X8/
    summary.json
    model.pt
    metrics.pt


In [ ]:
#8 experts, 64 hidden width (0.125 * 512 = 64)
#E50-X8
!python scripts/train_cifar10.py \
    --FF_layer SoftMoE \
    --epochs 50 \
    --batch_size 128 \
    --num_experts 8 \
    --hidden_mult 0.125 \
    --temperature 1.0 \
    --ckpt_root "{CKPT_ROOT}"


### Train SparseMoe (top2)
result: /content/drive/MyDrive/moe_project/checkpoints/SparseMoE/E50-X8-K2/
    summary.json
    model.pt
    metrics.pt


In [ ]:
#8 experts, 64 hidden width (0.125 * 512 = 64), top2
#E50-X8-K2
!python scripts/train_cifar10.py \
    --FF_layer SparseMoE \
    --epochs 50 \
    --batch_size 128 \
    --num_experts 8 \
    --hidden_mult 0.125 \
    --sparsemoe_k 2 \
    --ckpt_root "{CKPT_ROOT}"


### Run visualizations:

In [ ]:
import torch
from moe.utils.helpers import (
    plot_expert_utilization,
    plot_gating_entropy,
    plot_expert_probs_by_class,
    plot_expert_probs_heatmap,
    plot_expert_utilization_snapshot,
    plot_expert_load_over_epochs,
    plot_expert_load_snapshot,
)

ckpt_root = CKPT_ROOT  # same as before

paths = {
    "Dense":    os.path.join(ckpt_root, "Dense",    "E50",          "metrics.pt"),
    "SoftMoE":  os.path.join(ckpt_root, "SoftMoE",  "E50-X4",       "metrics.pt"),
    "SparseMoE":os.path.join(ckpt_root, "SparseMoE","E50-X4-K2",    "metrics.pt"),
}

history = {name: torch.load(path) for name, path in paths.items()}
history.keys()


In [ ]:
#TRAIN/VAL curves
import matplotlib.pyplot as plt
import numpy as np

for name, h in history.items():
    train_acc = np.array(h["train_acc"])
    val_acc   = np.array(h["val_acc"])

    plt.figure(figsize=(6,4))
    plt.plot(train_acc * 100, label="train")
    plt.plot(val_acc * 100, label="val")
    plt.xlabel("epoch")
    plt.ylabel("accuracy (%)")
    plt.title(f"{name}: train/val accuracy")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()


In [ ]:
#Gating stats for spars/softmoe
soft_hist   = history["SoftMoE"]
sparse_hist = history["SparseMoE"]

# Expert utilization over epochs
plot_expert_utilization(soft_hist,   ff_layer="SoftMoE")
plot_expert_utilization(sparse_hist, ff_layer="SparseMoE")

# Gating entropy over epochs
plot_gating_entropy(soft_hist,   ff_layer="SoftMoE")
plot_gating_entropy(sparse_hist, ff_layer="SparseMoE")

# If you logged per-class expert probabilities (class_expert_mean)
if "class_expert_mean" in soft_hist:
    plot_expert_probs_heatmap(
        soft_hist["class_expert_mean"],
        class_names=soft_hist.get("class_names", None),
        ff_layer="SoftMoE",
    )

if "class_expert_mean" in sparse_hist:
    plot_expert_probs_heatmap(
        sparse_hist["class_expert_mean"],
        class_names=sparse_hist.get("class_names", None),
        ff_layer="SparseMoE",
    )
